# 02 - Segmentación de un frame con SAM 3

Este notebook valida la inferencia de SAM 3 sobre un solo frame del video de prueba.

Objetivos:

1. Verificar el ambiente de ejecución.
2. Cargar un frame extraído del clip corto.
3. Inicializar SAM 3.
4. Ejecutar segmentación con prompts de texto.
5. Visualizar máscaras sobre el frame.
6. Guardar resultados preliminares en `outputs/figures/`.

Este notebook no procesa video completo. Primero se valida la segmentación en una sola imagen.


In [ ]:
from pathlib import Path
import sys
import os

import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memoria total [GB]:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


In [ ]:
# Rutas principales.
# Si Jupyter fue abierto desde la raíz del repositorio, esta ruta debe funcionar.
PROJECT_ROOT = Path.cwd()

# Si por alguna razón el notebook se abrió desde notebooks/, ajustar automáticamente.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FRAME_DIR = PROJECT_ROOT / "data" / "frames" / "test_20s"
OUTPUT_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", PROJECT_ROOT)
print("Carpeta de frames:", FRAME_DIR)

frames = sorted(FRAME_DIR.glob("*.jpg"))

print("Frames encontrados:", len(frames))

if len(frames) == 0:
    raise FileNotFoundError(f"No se encontraron frames en: {FRAME_DIR}")

FRAME_PATH = frames[0]
print("Frame seleccionado:", FRAME_PATH)


In [ ]:
# Cargar y visualizar el frame seleccionado.

image_bgr = cv2.imread(str(FRAME_PATH))

if image_bgr is None:
    raise RuntimeError(f"No se pudo leer la imagen: {FRAME_PATH}")

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

print("Dimensiones del frame:", image_rgb.shape)

plt.figure(figsize=(10, 6))
plt.imshow(image_rgb)
plt.axis("off")
plt.title(f"Frame seleccionado: {FRAME_PATH.name}")
plt.show()


## Inicialización de SAM 3

La siguiente celda inicializa el modelo de imagen de SAM 3 y su procesador. La primera ejecución puede tardar porque puede descargar pesos del modelo desde Hugging Face.

Si aparece un error de acceso, revisa que:

1. Iniciaste sesión con `hf auth login`.
2. Tu cuenta tiene acceso autorizado al modelo de SAM 3 en Hugging Face.
3. El ambiente activo es `futbotmx-sam3`.


In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONFIDENCE_THRESHOLD = 0.30

if DEVICE == "cuda":
    major = torch.cuda.get_device_properties(0).major
    if major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

print("Dispositivo usado:", DEVICE)

# Algunas versiones aceptan device como argumento; otras no.
try:
    model = build_sam3_image_model(device=DEVICE)
except TypeError:
    model = build_sam3_image_model()

processor = Sam3Processor(model, confidence_threshold=CONFIDENCE_THRESHOLD)

print("SAM 3 inicializado correctamente.")


In [ ]:
def to_numpy(x):
    """Convierte tensores/listas de SAM 3 a numpy."""
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def summarize_sam_output(output):
    """Imprime un resumen básico del resultado de SAM 3."""
    print("Claves del resultado:", list(output.keys()))

    for key in ["masks", "boxes", "scores"]:
        if key in output:
            arr = to_numpy(output[key])
            print(f"{key}: shape={None if arr is None else arr.shape}, dtype={None if arr is None else arr.dtype}")


def prepare_masks(masks):
    """Normaliza máscaras a forma [N, H, W]."""
    masks_np = to_numpy(masks)

    if masks_np is None:
        return np.empty((0, image_rgb.shape[0], image_rgb.shape[1]), dtype=bool)

    masks_np = np.asarray(masks_np)

    if masks_np.size == 0:
        return np.empty((0, image_rgb.shape[0], image_rgb.shape[1]), dtype=bool)

    # Casos frecuentes: [N, 1, H, W] o [N, H, W].
    if masks_np.ndim == 4 and masks_np.shape[1] == 1:
        masks_np = masks_np[:, 0, :, :]
    elif masks_np.ndim == 2:
        masks_np = masks_np[None, :, :]

    return masks_np.astype(bool)


def show_masks(image_rgb, masks, title="", max_masks=20):
    """Muestra máscaras superpuestas sobre la imagen."""
    masks_np = prepare_masks(masks)

    plt.figure(figsize=(10, 6))
    plt.imshow(image_rgb)

    if masks_np.shape[0] == 0:
        plt.title(title + " | Sin máscaras")
        plt.axis("off")
        plt.show()
        return

    for i, mask in enumerate(masks_np[:max_masks]):
        overlay = np.zeros((*mask.shape, 4), dtype=float)
        overlay[..., 3] = mask * 0.45
        plt.imshow(overlay)

    plt.title(f"{title} | Máscaras mostradas: {min(masks_np.shape[0], max_masks)}")
    plt.axis("off")
    plt.show()


In [ ]:
# Prueba inicial con un solo prompt.
# Puedes cambiar el prompt si el resultado no es útil.

PROMPT = "soccer ball"

image_pil = Image.open(FRAME_PATH).convert("RGB")

with torch.inference_mode():
    state = processor.set_image(image_pil)
    output = processor.set_text_prompt(state=state, prompt=PROMPT)

summarize_sam_output(output)
show_masks(image_rgb, output.get("masks"), title=f"Prompt: {PROMPT}")


In [ ]:
# Guardar una figura preliminar del resultado.

def save_overlay_figure(image_rgb, masks, output_path, title=""):
    masks_np = prepare_masks(masks)

    plt.figure(figsize=(10, 6))
    plt.imshow(image_rgb)

    for mask in masks_np[:20]:
        overlay = np.zeros((*mask.shape, 4), dtype=float)
        overlay[..., 3] = mask * 0.45
        plt.imshow(overlay)

    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close()

    print("Figura guardada en:", output_path)


safe_prompt = PROMPT.replace(" ", "_").replace("/", "_")
output_path = OUTPUT_FIGURES_DIR / f"sam3_{safe_prompt}_{FRAME_PATH.stem}.png"

save_overlay_figure(
    image_rgb=image_rgb,
    masks=output.get("masks"),
    output_path=output_path,
    title=f"SAM 3 | Prompt: {PROMPT}"
)


In [ ]:
# Prueba comparativa con varios prompts.
# Esta celda ayuda a identificar qué prompts funcionan mejor para el dominio del reto.

prompts = [
    "soccer ball",
    "robot soccer player",
    "playing field",
    "goal",
    "blue robot",
    "red robot",
]

results = {}

image_pil = Image.open(FRAME_PATH).convert("RGB")

with torch.inference_mode():
    base_state = processor.set_image(image_pil)

    for prompt in prompts:
        print("\nPrompt:", prompt)
        try:
            out = processor.set_text_prompt(state=base_state, prompt=prompt)
            results[prompt] = out
            summarize_sam_output(out)
        except Exception as exc:
            print(f"Error con prompt '{prompt}': {exc}")
            results[prompt] = None


In [ ]:
# Visualizar resultados por prompt.

for prompt, out in results.items():
    if out is None:
        continue

    show_masks(
        image_rgb=image_rgb,
        masks=out.get("masks"),
        title=f"Prompt: {prompt}",
        max_masks=20,
    )


In [ ]:
# Guardar figuras de todos los prompts exitosos.

for prompt, out in results.items():
    if out is None:
        continue

    safe_prompt = prompt.replace(" ", "_").replace("/", "_")
    output_path = OUTPUT_FIGURES_DIR / f"sam3_{safe_prompt}_{FRAME_PATH.stem}.png"

    save_overlay_figure(
        image_rgb=image_rgb,
        masks=out.get("masks"),
        output_path=output_path,
        title=f"SAM 3 | Prompt: {prompt}"
    )


## Siguiente paso

Si este notebook logra producir máscaras útiles, el siguiente hito será convertir esas máscaras en datos geométricos:

- Área.
- Caja delimitadora.
- Centroide.
- Clase asociada al prompt.
- Frame.
- Tiempo.

Con esos datos se podrá construir el primer archivo `tracks.csv` y comenzar el tracking temporal.
